# Data Description
- Author: 李芸榛（JenniferjjLee）
- Date first created: 20261108（v6 資料檔產生日；本文件取代 2026-10-20 版的舊 78 站文件，舊版見 git 歷史）
- Last updated: 20261108（改為描述 v7：修正截斷站名 `捷運科技大樓站(台北教育大學`；同日改用 109 站的新版 `stations_in_field`，臺大人文館改用正式 StationID `500119097`，並納入羅斯福路三段283巷；兩個年度檔改為 Parquet）
- Background: 臺北市 YouBike 2.0 票證刷卡交易中，借車站或還車站任一端位於臺大周邊研究範圍的旅次，2024、2025 年各一檔。研究範圍 = `data/processed/stations_in_field.csv`（2026-11-08 版）的 109 站，含 2025 年才啟用的 `臺大人文館` 與 `羅斯福路三段283巷`。篩選採站名精確比對（OR）；不去重、不補值。唯一的站名修正：原始資料 2024-01～2025-01 把 `捷運科技大樓站(台北教育大學)`（`500101166`）寫成少了右括號的 `捷運科技大樓站(台北教育大學`，篩選前先改回完整站名（v6 把它當成範圍外，另一端也在範圍外的交易則整筆漏掉）。站名一律加前綴 `YouBike2.0_`，並附借、還兩端的 StationID，以及起訖分組欄 `trip_group`（a／b／c／d）。
- Unit of observation: 一筆來源交易紀錄。資料沒有交易 ID，9 欄組合也不唯一（見 Summary Stats 的完全重複列），不宣稱任何唯一鍵。
- Sample period: 依來源檔月份分年：2024 檔 = 202401–202412 借車，2025 檔 = 202501–202512 借車（借車時間必在該年度內；還車可跨年，最晚到 2034-08-26，見 Known issues）。
- Known issues:
  - **時間只精確到小時**：`rent_time`、`return_time` 的分秒全為 `00:00`，不能用兩者相減求騎乘時間，要用 `rent`。
  - **2025 年新站**：臺大人文館（`500119097`，2025-02-04 起）和羅斯福路三段283巷（`500101265`，2025-10-10 起）只有 2025 年有交易；2025 年分別有 99,221 列、13,337 列的一端是這兩站，2024 年都是 0。跨年比較需要控制站點數。（前版臺大人文館暫用 `-2`、283巷算範圍外，本版已改正。）
  - **`-1` 的含意**：範圍外站點，也包含身分不明的空白站名（2024：1 格；2025：3 格）與亂碼 token（2024：820 格；2025：108 格；以 `<UNRESOLVED_UTF8_HEX:...>` 可逆保留原 bytes，不是修復後的站名）。
  - **車種**：2024-01～10 的原始檔沒有 `bike_type` 欄，輸出為空字串（5,993,280 列，占 2024 年 77.06%），不代表一般車；2025 年沒有空白。
  - **時長異常**：≤ 1 分鐘（2024：5,232；2025：20,079）、> 4 小時（2024：7,423；2025：7,477）；2024 年最長一筆約 10 年（5,258,885.15 分鐘）。平均數與標準差受極端值影響。
  - **還車時間異常**：還車年份不等於來源年度的有 2024 年 308 列、2025 年 98 列，多數是正常跨年；但 2024 年有 1 列還車在 2034-08-26，明顯錯誤。借還小時差與 `rent` 對不上的有 2024 年 7 列、2025 年 9 列。
  - **完全重複列**：所有欄全同（含 `rent` 秒數）2024 年 73,930 列、2025 年 79,491 列（以 `keep=False` 計，含每組第一筆）。依原規則保留，是否去重需在分析時決定。
  - **`trip_group` 的 d 組只在 2025-08 以後出現**：2025-07 以前，資料來源就沒有 5 分鐘內的同站還車紀錄（同站借還最短幾乎都是 300–301 秒）；2025-08 起才出現 1–6 秒的紀錄（d 組共 2,729 列）。所以 d 組不能做跨期比較；2025-08 以後的 c 組也多了 3–5 分鐘的同站借還。
- Definition for each variable:
  - `rent_time`：借車時間，`YYYY-MM-DD HH:00:00`（小時精度）。
  - `rent_station`：借車站名，加前綴 `YouBike2.0_`；空白與亂碼 token 保持原樣。
  - `rent_station_id`：借車站的 StationID。範圍內 109 站為 9 碼 ID；其他為 `-1`。
  - `return_time`：還車時間，格式同 `rent_time`。
  - `return_station`：還車站名，處理同 `rent_station`。
  - `return_station_id`：還車站的 StationID，規則同 `rent_station_id`。
  - `rent`：騎乘時長字串 `H:MM:SS`（小時可超過 2 位），精確到秒；官方定義待確認。
  - `bike_type`：車種，`一般車`、`電輔車`，或空字串（來源未提供）。
  - `infodate`：交易日期，`YYYY-MM-DD`，與借車日期相同。
  - `trip_group`：起訖分組（四組互斥，先判斷 d；「台大」= 研究範圍 109 站）：
    - `a` 外借台大還（流入）：借車站 ID = `-1`，還車站 ID ≠ `-1`。
    - `b` 台大借外還（流出）：借車站 ID ≠ `-1`，還車站 ID = `-1`。
    - `c` 台大借台大還（區內）：兩端 ID 都 ≠ `-1`，且不屬於 d。
    - `d` 無效還車：借還同站，且 `rent` 時長 ≤ 180 秒（3 分鐘；組員 2026-11-08 依同站借還時長分布決定）。
- Data path:
  - `data/processed/youbike_traffic/YouBike_traffic_2024_v7.parquet`（7,776,981 列 × 10 欄）
  - `data/processed/youbike_traffic/YouBike_traffic_2025_v7.parquet`（10,253,912 列 × 10 欄）
  - 兩個年度檔為 Parquet（zstd 壓縮，約 45 MB／60 MB），低於 GitHub 100 MB 上限，可以 commit。10 欄都是字串，空白值為空字串 `""`（不是 null），與先前 CSV 版逐列相同；CSV 版已於 2026-11-08 刪除。原本的 100 列範例檔也已刪除。
- Source data: `data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/YYYYMM_YouBike2.0票證刷卡資料.csv`（202401–202512，24 個月檔，合計 133,966,680 筆）與 `data/processed/stations_in_field.csv`（2026-11-08 版，109 站）。
- Processing code: [data/code/youbike_traffic_station_id_v7.ipynb](../../code/youbike_traffic_station_id_v7.ipynb)（讀檔函式沿用 `data/raw/2024-25YouBike2.0票證刷卡資料/YouBike_traffic_raw.py`）。
- Role: `analysis`。
- Verification status: 2026-11-08 在全新的 in-process IPython shell 由上而下執行本 notebook（本機沒有 Jupyter），三個檔案的 SHA-256、欄位、列數、OR 篩選、StationID 規則、`trip_group` 分組規則、時間格式、日期一致性、車種允許值與範例檔內容全部通過（見 Assertions）；同日改為描述 v7 後重跑，全部通過，並確認 v7 已沒有截斷站名；改用 109 站清單後再重跑一次，全部通過；同日年度檔改為 Parquet 後重跑，全部通過；刪除範例檔並移除相關檢查後再重跑，全部通過。資料品質問題已列於 Known issues，只計數、沒有修正。
- SHA-256:
  - 2024：`5739637b670bceb34ed2e7271cb78da56f9ea56da310bfb53ce8dccaadd5e119`
  - 2025：`9b1e3beb9d0e4a164c2ad6662764150768452de126be46329566d6030c0c61d9`
- Downstream analysis input: `data/processed/youbike_traffic/YouBike_traffic_2024_v7.parquet`、`data/processed/youbike_traffic/YouBike_traffic_2025_v7.parquet`（與 Data path 相同）


In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


import numpy as np
from IPython.display import display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Change for Your Own Data

下方設定已綁定本文件描述的三個實際檔案與 `stations_in_field.csv`（只用於核對 StationID）；路徑不存在或 SHA-256 不符就停止，不會改讀其他版本。所有欄位以字串讀入、空字串保留，與產生程式的寫出方式一致。

In [2]:
# Exact files documented here (all produced by data/code/youbike_traffic_station_id_v7.ipynb).
OUT_DIR = "data/processed/youbike_traffic"
INPUTS = {
    2024: {"path": f"{OUT_DIR}/YouBike_traffic_2024_v7.parquet", "rows": 7_776_981,
           "sha256": "5739637b670bceb34ed2e7271cb78da56f9ea56da310bfb53ce8dccaadd5e119"},
    2025: {"path": f"{OUT_DIR}/YouBike_traffic_2025_v7.parquet", "rows": 10_253_912,
           "sha256": "9b1e3beb9d0e4a164c2ad6662764150768452de126be46329566d6030c0c61d9"},
}
# Station list used to assign StationID (needed for the ID assertions only).
STATIONS = {"path": "data/processed/stations_in_field.csv",
            "sha256": "74dadd7afe58f0b69b0a511f234dc85388e64290d5db420b8426767bad637f9e"}
EXPECTED_COLUMNS = ["rent_time", "rent_station", "rent_station_id", "return_time", "return_station",
                    "return_station_id", "rent", "bike_type", "infodate", "trip_group"]
TRIP_GROUPS = {"a": "外借台大還（流入）", "b": "台大借外還（流出）", "c": "台大借台大還（區內，不含 d）",
               "d": "無效還車（同站借還且時長 ≤ 3 分鐘）"}
INVALID_MAX_SECONDS = 180
PREFIX = "YouBike2.0_"
OUTSIDE_ID = "-1"
# Stations first appearing in 2025 (added in the 2026-11-08 station list): 2024 has 107 stations with trips, 2025 has 109.
NEW_2025_STATIONS = {"臺大人文館": "500119097", "羅斯福路三段283巷": "500101265"}
EXPECTED_STATIONS_SEEN = {2024: 107, 2025: 109}
TOKEN_PREFIX = "<UNRESOLVED_UTF8_HEX:"
TEXT_READ = {"dtype": str, "keep_default_na": False, "na_filter": False, "encoding": "utf-8-sig"}

## Summary

先核對每個檔案的 SHA-256 再讀入，並列出列數與欄位型別。以 `_` 開頭的欄位是為了核對而暫時衍生（時間解析、時長分鐘數、旅次類型），不寫回檔案。

In [3]:
# Verify bytes before loading, then read every column as text (blank strings stay "").
frames = {}
for year, info in INPUTS.items():
    path = project_file(info["path"], "processed")
    info["fingerprint"] = verify_input_sha256(path, info["sha256"])
    frames[year] = pd.read_parquet(path)  # every column is text; blanks are ""
    print({"data_path": info["path"], "rows": len(frames[year]),
           "schema": {c: str(t) for c, t in frames[year].dtypes.items()}, "sha256": info["fingerprint"]})
stations_path = project_file(STATIONS["path"], "processed")
verify_input_sha256(stations_path, STATIONS["sha256"])
stations = pd.read_csv(stations_path, dtype=str, keep_default_na=False)
STATION_ID = dict(stations[["StationNameZh_tw", "StationID"]].drop_duplicates().itertuples(index=False))
assert len(STATION_ID) == 109
for name, station_id in NEW_2025_STATIONS.items():
    assert STATION_ID[PREFIX + name] == station_id, name
print({"station names with ID": len(STATION_ID)})

# Derived columns for checks only (the data files are not modified).
for year, df in frames.items():
    df["_rent_dt"] = pd.to_datetime(df["rent_time"], format="%Y-%m-%d %H:%M:%S")
    df["_return_dt"] = pd.to_datetime(df["return_time"], format="%Y-%m-%d %H:%M:%S")
    rent = df["rent"]
    df["_duration_min"] = (rent.str[:-6].astype(np.int64) * 3600 + rent.str[-5:-3].astype(np.int64) * 60
                           + rent.str[-2:].astype(np.int64)) / 60
    rent_in, return_in = df["rent_station_id"] != OUTSIDE_ID, df["return_station_id"] != OUTSIDE_ID
    df["_trip_type"] = np.select([rent_in & return_in, rent_in], ["區內", "流出"], "流入")
    short_same = (df["rent_station"] == df["return_station"]) & (df["_duration_min"] * 60 <= INVALID_MAX_SECONDS)
    df["_group_rule"] = np.select([short_same, ~rent_in & return_in, rent_in & ~return_in, rent_in & return_in],
                                  ["d", "a", "b", "c"], default="?")

{'data_path': 'data/processed/youbike_traffic/YouBike_traffic_2024_v7.parquet', 'rows': 7776981, 'schema': {'rent_time': 'object', 'rent_station': 'object', 'rent_station_id': 'object', 'return_time': 'object', 'return_station': 'object', 'return_station_id': 'object', 'rent': 'object', 'bike_type': 'object', 'infodate': 'object', 'trip_group': 'object'}, 'sha256': '5739637b670bceb34ed2e7271cb78da56f9ea56da310bfb53ce8dccaadd5e119'}


{'data_path': 'data/processed/youbike_traffic/YouBike_traffic_2025_v7.parquet', 'rows': 10253912, 'schema': {'rent_time': 'object', 'rent_station': 'object', 'rent_station_id': 'object', 'return_time': 'object', 'return_station': 'object', 'return_station_id': 'object', 'rent': 'object', 'bike_type': 'object', 'infodate': 'object', 'trip_group': 'object'}, 'sha256': '9b1e3beb9d0e4a164c2ad6662764150768452de126be46329566d6030c0c61d9'}
{'station names with ID': 109}


### Sample

In [4]:
for year, df in frames.items():
    print(f"{year} 年：依檔案順序的前 10 筆（原始字串）")
    display(df[EXPECTED_COLUMNS].head(10))

2024 年：依檔案順序的前 10 筆（原始字串）


,rent_time,rent_station,rent_station_id,return_time,return_station,return_station_id,rent,bike_type,infodate,trip_group
0,2024-01-22 08:00:00,YouBike2.0_新生和平路口東北側,-1,2024-01-22 09:00:00,YouBike2.0_新生南路三段66號前,500101009,00:07:02,,2024-01-22,a
1,2024-01-22 12:00:00,YouBike2.0_臺大小福樓東側,500119050,2024-01-22 12:00:00,YouBike2.0_捷運臺電大樓站(5號出口),-1,00:12:21,,2024-01-22,b
2,2024-01-22 21:00:00,YouBike2.0_捷運公館站(2號出口),500101022,2024-01-22 22:00:00,YouBike2.0_臺灣科技大學側門,-1,00:19:06,,2024-01-22,b
3,2024-01-22 23:00:00,YouBike2.0_捷運公館站(2號出口),500101022,2024-01-23 00:00:00,YouBike2.0_臺灣科技大學側門,-1,00:26:58,,2024-01-22,b
4,2024-01-22 18:00:00,YouBike2.0_臺大工綜館南側,500119052,2024-01-22 18:00:00,YouBike2.0_臺大椰林小舖,500119070,00:04:00,,2024-01-22,c
5,2024-01-22 15:00:00,YouBike2.0_臺大新生教室南側,500119083,2024-01-22 15:00:00,YouBike2.0_臺大新生教室南側,500119083,00:23:07,,2024-01-22,c
6,2024-01-22 17:00:00,YouBike2.0_臺大卓越研究大樓,500119006,2024-01-22 17:00:00,YouBike2.0_臺大黑森林西側,500119088,00:11:16,,2024-01-22,c
7,2024-01-18 21:00:00,YouBike2.0_濟南紹興路口,-1,2024-01-18 21:00:00,YouBike2.0_辛亥泰順街口,500101094,00:17:26,,2024-01-18,a
8,2024-01-18 11:00:00,YouBike2.0_林森仁愛路口,-1,2024-01-18 11:00:00,YouBike2.0_臺大第一活動中心西南側,500119075,00:22:55,,2024-01-18,a
9,2024-01-18 16:00:00,YouBike2.0_林森仁愛路口,-1,2024-01-18 16:00:00,YouBike2.0_新生南路三段82號前,500101010,00:22:05,,2024-01-18,a


2025 年：依檔案順序的前 10 筆（原始字串）


,rent_time,rent_station,rent_station_id,return_time,return_station,return_station_id,rent,bike_type,infodate,trip_group
0,2025-01-13 15:00:00,YouBike2.0_捷運古亭站(8號出口),-1,2025-01-13 21:00:00,YouBike2.0_新生南路三段52號前,500101008,05:34:25,電輔車,2025-01-13,a
1,2025-01-13 01:00:00,YouBike2.0_辛亥路一段30號前,500101012,2025-01-13 03:00:00,YouBike2.0_捷運公館站(4號出口),500106004,01:50:15,一般車,2025-01-13,c
2,2025-01-13 11:00:00,YouBike2.0_臺大土木研究大樓前,500101035,2025-01-13 12:00:00,YouBike2.0_立人國小,-1,00:15:08,一般車,2025-01-13,b
3,2025-01-13 12:00:00,YouBike2.0_臺大男一舍前,500101032,2025-01-13 12:00:00,YouBike2.0_捷運公館站(2號出口),500101022,00:06:22,一般車,2025-01-13,c
4,2025-01-13 17:00:00,YouBike2.0_和平泰順街口,-1,2025-01-13 17:00:00,YouBike2.0_羅斯福路三段245號前,500101016,00:04:40,電輔車,2025-01-13,a
5,2025-01-13 15:00:00,YouBike2.0_臺大卓越研究大樓,500119006,2025-01-13 16:00:00,YouBike2.0_臺大立體機車停車場,500119051,00:35:58,一般車,2025-01-13,c
6,2025-01-13 16:00:00,YouBike2.0_臺大土木研究大樓前,500101035,2025-01-13 17:00:00,YouBike2.0_捷運萬隆站(2號出口),-1,00:52:45,一般車,2025-01-13,b
7,2025-01-13 15:00:00,YouBike2.0_捷運公館站(2號出口),500101022,2025-01-13 16:00:00,YouBike2.0_臺大鹿鳴堂東側,500119061,00:32:42,一般車,2025-01-13,c
8,2025-01-13 15:00:00,YouBike2.0_臺大小福樓東側,500119050,2025-01-13 16:00:00,YouBike2.0_臺大小福樓東側,500119050,00:38:56,一般車,2025-01-13,c
9,2025-01-13 13:00:00,YouBike2.0_臺大工綜館南側,500119052,2025-01-13 13:00:00,YouBike2.0_臺大二號館,500119065,00:03:08,一般車,2025-01-13,c


### Summary Stats

In [5]:
# Summary stats. Denominator for every percentage = all rows of that year. "Missing" = empty string.
def fmt_int(x): return f"{int(x):,}"
def fmt2(x): return f"{x:,.2f}"

rows = []
for year, df in frames.items():
    n = len(df)
    for col in ["rent_station", "rent_station_id", "return_station", "return_station_id", "bike_type", "infodate", "trip_group"]:
        s = df[col]
        top = s[s != ""].value_counts().head(10)
        rows.append({"年度": year, "變數": col, "N": fmt_int(n), "空字串 N": fmt_int((s == "").sum()),
                     "不同值數": fmt_int(s.nunique()),
                     "前 10 名（筆數，占比）": "；".join(f"{k}（{v:,}，{v / n:.2%}）" for k, v in top.items())})
print("表 1　文字／類別欄位（空字串視為缺值；前 10 名占比的分母 = 該年度全部列）")
display(pd.DataFrame(rows).set_index(["年度", "變數"]))

num = {}
for year, df in frames.items():
    d = df["_duration_min"]
    num[f"{year} 騎乘時長（分鐘，由 rent 換算）"] = {
        "N": fmt_int(d.notna().sum()), "缺值 N": fmt_int(d.isna().sum()), "平均數": fmt2(d.mean()),
        "標準差": fmt2(d.std()), "最小值": fmt2(d.min()), "P25": fmt2(d.quantile(.25)), "中位數": fmt2(d.median()),
        "P75": fmt2(d.quantile(.75)), "最大值": fmt2(d.max())}
print("表 2　數值摘要（rent 原為 H:MM:SS 字串；換算只供描述，不回寫）")
display(pd.DataFrame(num).T)

quality = {}
for year, df in frames.items():
    gap_h = (df["_return_dt"] - df["_rent_dt"]).dt.total_seconds() / 3600
    lo = np.floor(df["_duration_min"] / 60)
    q = {
        "列數": len(df),
        "rent_time 範圍": f"{df['_rent_dt'].min():%Y-%m-%d %H} 時～{df['_rent_dt'].max():%Y-%m-%d %H} 時",
        "return_time 範圍": f"{df['_return_dt'].min():%Y-%m-%d %H} 時～{df['_return_dt'].max():%Y-%m-%d %H} 時",
        "還車年份 ≠ 來源年度": int((df["_return_dt"].dt.year != year).sum()),
        "還車年份 > 來源年度 + 1": int((df["_return_dt"].dt.year > year + 1).sum()),
        "還車早於借車": int((gap_h < 0).sum()),
        "借還小時差與 rent 不一致": int((~((gap_h == lo) | (gap_h == lo + 1))).sum()),
        "時長 ≤ 1 分鐘": int((df["_duration_min"] <= 1).sum()),
        "時長 > 4 小時": int((df["_duration_min"] > 240).sum()),
        "完全重複列（所有欄全同）": int(df.duplicated(EXPECTED_COLUMNS, keep=False).sum()),
        "站名為亂碼 token（借或還）": int((df["rent_station"].str.startswith(TOKEN_PREFIX)
                                     | df["return_station"].str.startswith(TOKEN_PREFIX)).sum()),
        **{f"一端為{name}（{sid}）": int(((df["rent_station_id"] == sid) | (df["return_station_id"] == sid)).sum())
           for name, sid in NEW_2025_STATIONS.items()},
        "區內旅次": int((df["_trip_type"] == "區內").sum()),
        "流出旅次（範圍內借、範圍外還）": int((df["_trip_type"] == "流出").sum()),
        "流入旅次（範圍外借、範圍內還）": int((df["_trip_type"] == "流入").sum()),
    }
    quality[year] = {k: (fmt_int(v) if isinstance(v, (int, np.integer)) else v) for k, v in q.items()}
print("表 3　時間範圍、資料品質與旅次類型（只計數，不刪除）")
display(pd.DataFrame(quality))

group_rows = {}
for year, df in frames.items():
    counts = df["trip_group"].value_counts()
    for g, label in TRIP_GROUPS.items():
        group_rows.setdefault(f"{g} {label}", {})[year] = int(counts.get(g, 0))
group_table = pd.DataFrame(group_rows).T
group_table["合計"] = group_table.sum(axis=1)
group_table["占全部"] = (group_table["合計"] / group_table["合計"].sum()).map(lambda x: f"{x:.2%}")
for col in (2024, 2025, "合計"):
    group_table[col] = group_table[col].map(fmt_int)
print("表 4　trip_group 各組列數")
display(group_table)
d_month = pd.concat([df.loc[df["trip_group"] == "d", "infodate"].str[:7] for df in frames.values()]).value_counts().sort_index()
print("表 5　d 組各月列數（2025-08 以前資料來源沒有 5 分鐘內的同站還車）")
display(d_month.rename("列數").to_frame().T)

表 1　文字／類別欄位（空字串視為缺值；前 10 名占比的分母 = 該年度全部列）


N      空字串 N   不同值數                                      前 10 名（筆數，占比）
年度   變數                                                                                                
2024 rent_station        7,776,981          0  1,393  YouBike2.0_捷運公館站(2號出口)（498,239，6.41%）；YouBike2...
     rent_station_id     7,776,981          0    108  -1（1,710,313，21.99%）；500101022（498,239，6.41%）；...
     return_station      7,776,981          1  2,226  YouBike2.0_捷運公館站(2號出口)（518,727，6.67%）；YouBike2...
     return_station_id   7,776,981          0    108  -1（1,858,538，23.90%）；500101022（518,727，6.67%）；...
     bike_type           7,776,981  5,993,280      3            一般車（1,756,806，22.59%）；電輔車（26,895，0.35%）
     infodate            7,776,981          0    364  2024-09-03（42,182，0.54%）；2024-09-20（41,894，0.5...
     trip_group          7,776,981          0      3  c（4,208,130，54.11%）；b（1,858,538，23.90%）；a（1,71...
2025 rent_station       10,253,912          1  1,567  YouBike2.0_捷運公館站(2號出口)（561,546，5.48%）；YouBike2...
     rent_station_id    10,253,912          0    110  -1（2,210,188，21.55%）；500101022（561,546，5.48%）；...
     return_station     10,253,912          2  2,525  YouBike2.0_捷運公館站(2號出口)（606,903，5.92%）；YouBike2...
     return_station_id  10,253,912          0    110  -1（2,403,320，23.44%）；500101022（606,903，5.92%）；...
     bike_type          10,253,912          0      2          一般車（10,022,917，97.75%）；電輔車（230,995，2.25%）
     infodate           10,253,912          0    365  2025-09-03（47,001，0.46%）；2025-09-10（46,811，0.4...
     trip_group         10,253,912          0      4  c（5,637,675，54.98%）；b（2,403,320，23.44%）；a（2,21...

表 2　數值摘要（rent 原為 H:MM:SS 字串；換算只供描述，不回寫）


,N,缺值 N,平均數,標準差,最小值,P25,中位數,P75,最大值
2024 騎乘時長（分鐘，由 rent 換算）,"7,776,981",0,13.70,"1,886.84",0.00,4.78,7.58,13.87,"5,258,885.15"
2025 騎乘時長（分鐘，由 rent 換算）,"10,253,912",0,11.86,43.54,0.00,4.55,7.23,13.00,"54,463.80"


表 3　時間範圍、資料品質與旅次類型（只計數，不刪除）


,2024,2025
列數,"7,776,981","10,253,912"
rent_time 範圍,2024-01-01 00 時～2024-12-31 23 時,2025-01-01 00 時～2025-12-31 23 時
return_time 範圍,2024-01-01 00 時～2034-08-26 10 時,2025-01-01 00 時～2026-01-01 07 時
還車年份 ≠ 來源年度,308,98
還車年份 > 來源年度 + 1,1,0
還車早於借車,0,0
借還小時差與 rent 不一致,7,9
時長 ≤ 1 分鐘,"5,232","20,079"
時長 > 4 小時,"7,423","7,477"
完全重複列（所有欄全同）,"73,930","79,491"


表 4　trip_group 各組列數


,2024,2025,合計,占全部
a 外借台大還（流入）,"1,710,313","2,210,188","3,920,501",21.74%
b 台大借外還（流出）,"1,858,538","2,403,320","4,261,858",23.64%
c 台大借台大還（區內，不含 d）,"4,208,130","5,637,675","9,845,805",54.61%
d 無效還車（同站借還且時長 ≤ 3 分鐘）,0,"2,729","2,729",0.02%


表 5　d 組各月列數（2025-08 以前資料來源沒有 5 分鐘內的同站還車）


infodate,2025-08,2025-09,2025-10,2025-11,2025-12
列數,126,713,561,724,605


### Assertions

只檢查由產生程式規則支持的性質：檔案版本、欄位、列數、OR 篩選條件、StationID 指派規則（109 站真實 ID、其他 `-1`；2024 年 107 站、2025 年 109 站有交易）、時間精度、`infodate` 與借車日一致、借車年份與來源年度一致、`rent` 格式、車種允許值；另確認輸出中沒有截斷站名 `捷運科技大樓站(台北教育大學`。不要求沒有重複列，也不宣稱存在唯一鍵（資料沒有交易 ID）。

In [6]:
# Assertions grounded in the producing notebook's rules.
checks = []
for year, df in frames.items():
    info = INPUTS[year]
    assert list(df.columns[:10]) == EXPECTED_COLUMNS, f"{year}: schema differs"
    assert (df["trip_group"] == df["_group_rule"]).all(), f"{year}: trip_group differs from its rule"
    assert df["trip_group"].isin(list(TRIP_GROUPS)).all(), f"{year}: unexpected trip_group"
    assert (df.loc[df["trip_group"] == "d", "infodate"] >= "2025-08-01").all(), f"{year}: d before 2025-08"
    assert len(df) == info["rows"], f"{year}: row count differs from the processing record"
    for side in ("rent", "return"):
        names, ids = df[f"{side}_station"], df[f"{side}_station_id"]
        listed = names.isin(STATION_ID)
        assert (ids[listed] == names[listed].map(STATION_ID)).all(), f"{year} {side}: ID differs from station list"
        assert (ids[~listed] == OUTSIDE_ID).all(), f"{year} {side}: unlisted station without -1"
        assert ids.str.fullmatch(r"\d{9}|-1").all(), f"{year} {side}: ID is neither a 9-digit StationID nor -1"
        is_token = names.str.startswith(TOKEN_PREFIX)
        assert not (names == PREFIX + "捷運科技大樓站(台北教育大學").any(), f"{year} {side}: truncated station name left"
        assert names[(names != "") & ~is_token].str.startswith(PREFIX).all(), f"{year} {side}: missing prefix"
    assert ((df["rent_station_id"] != OUTSIDE_ID) | (df["return_station_id"] != OUTSIDE_ID)).all(), \
        f"{year}: a row has neither end in the field"
    real_ids = (set(df["rent_station_id"]) | set(df["return_station_id"])) - {OUTSIDE_ID}
    assert len(real_ids) == EXPECTED_STATIONS_SEEN[year], f"{year}: {len(real_ids)} stations appear"
    for col in ("rent_time", "return_time"):
        assert df[col].str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:00:00").all(), f"{year}: {col} not hour-precision"
    assert (df["infodate"] == df["rent_time"].str[:10]).all(), f"{year}: infodate differs from rental date"
    assert (df["_rent_dt"].dt.year == year).all(), f"{year}: rental outside the source year"
    assert df["rent"].str.fullmatch(r"\d+:[0-5]\d:[0-5]\d").all(), f"{year}: rent format"
    assert df["bike_type"].isin(["", "一般車", "電輔車"]).all(), f"{year}: unexpected bike_type"
    if year == 2024:
        early = df["infodate"] < "2024-11-01"
        assert (df.loc[early, "bike_type"] == "").all(), "2024-01..10 source files have no bike_type column"
    assert sha256_file(project_file(info["path"], "processed")) == info["fingerprint"], f"{year}: file changed"
    checks.append({"data_path": info["path"], "rows": len(df), "columns": 10, "stations with real ID": len(real_ids),
                   "sha256": info["fingerprint"], "status": "configured checks passed"})

for c in checks:
    print(c)

{'data_path': 'data/processed/youbike_traffic/YouBike_traffic_2024_v7.parquet', 'rows': 7776981, 'columns': 10, 'stations with real ID': 107, 'sha256': '5739637b670bceb34ed2e7271cb78da56f9ea56da310bfb53ce8dccaadd5e119', 'status': 'configured checks passed'}
{'data_path': 'data/processed/youbike_traffic/YouBike_traffic_2025_v7.parquet', 'rows': 10253912, 'columns': 10, 'stations with real ID': 109, 'sha256': '9b1e3beb9d0e4a164c2ad6662764150768452de126be46329566d6030c0c61d9', 'status': 'configured checks passed'}


### 本次文件核對的結論與限制

- 三個檔案都與產生程式的紀錄一致，所有 assertion 通過。`trip_group` 兩年合計：a 3,920,501、b 4,261,858、c 9,845,805、d 2,729 列。兩年的旅次類型：2024 年區內 4,208,130、流出 1,858,538、流入 1,710,313；2025 年區內 5,640,404、流出 2,403,320、流入 2,210,188。
- 與 v6 相比（改用 109 站清單前），2024 年多 117,154 列、2025 年多 9,675 列，全部來自截斷站名修正；兩端都不是 `500101166` 的列與 v6 完全相同（比對見產生程式的結果摘要）。
- 改用 109 站清單後：2024 年檔案逐位元組不變；2025 年多 7,681 列（283巷 與範圍外站之間的旅次），臺大人文館的 ID 由 `-2` 改為 `500119097`，一端是 283巷 且另一端在範圍內的旅次由 a／b 改歸 c；其餘列完全相同。
- 本文件只描述與核對，沒有修改任何資料。Known issues 中的時長異常、還車在 2034 年的錯誤列、完全重複列都還保留在檔案裡，分析前需決定處理方式。
- 舊版文件（2026-10-20）描述的是舊 78 站版本（`stage2_res/` 年度檔），該版資料已被取代並刪除。舊版提到的旁表漏列問題屬於 stage2 舊流程，不適用於本版。
